# MOD16A2 Consolidated Output Viewer

Visualize the consolidated MOD16A2 (ET) dataset for a selected timestep.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

# --- Configuration ---
DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
NC_PATH = DATASTORE / "mod16a2_v061" / "mod16a2_v061_2005_consolidated.nc"
TIMESTEP = 0  # Change this to view different 8-day composites


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime steps: {len(ds.time)}")
print(f"Selected: timestep {TIMESTEP} → {ds.time.values[TIMESTEP]}")


In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap, BoundaryNorm

et = ds["ET_500m"].isel(time=TIMESTEP)
qc_raw = ds["ET_QC_500m"].isel(time=TIMESTEP)
time_label = str(ds.time.values[TIMESTEP])[:10]
qc = qc_raw.values.astype(np.uint8)

# --- Plot 1: Raw ET ---
fig, ax = plt.subplots(figsize=(14, 6))
et.plot(ax=ax, cmap="YlGnBu", robust=True)
ax.set_title(f"ET (raw, all pixels) — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# --- Decode all QC bit fields ---
# Bits 0-1: Overall QC
overall_qc = qc & 0b11
# Bit 2: Cloudiness
cloud = (qc >> 2) & 0b1
# Bit 3: Elevation source
elev_src = (qc >> 3) & 0b1
# Bits 4-7: Land cover class
land_cover = (qc >> 4) & 0b1111

# --- Plot 2: Bits 0-1 — Overall QC ---
qc_da = xr.DataArray(overall_qc, dims=et.dims, coords=et.coords)
cmap_qc = ListedColormap(["#2ecc71", "#f39c12", "#e74c3c", "#95a5a6"])
norm_qc = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap_qc.N)
fig, ax = plt.subplots(figsize=(14, 6))
qc_da.plot(ax=ax, cmap=cmap_qc, norm=norm_qc, add_colorbar=False)
cbar = plt.colorbar(ax.collections[0], ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(["Good", "Marginal", "Poor", "Fill"])
ax.set_title(f"Bits 0-1: Overall QC — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

for val, label in {0: "Good", 1: "Marginal", 2: "Poor", 3: "Fill"}.items():
    pct = (overall_qc == val).sum() / overall_qc.size * 100
    print(f"  QC={val} ({label}): {pct:.1f}%")

# --- Plot 3: Bit 2 — Cloudiness ---
cloud_da = xr.DataArray(cloud, dims=et.dims, coords=et.coords)
cmap_cloud = ListedColormap(["#3498db", "#ecf0f1"])
norm_cloud = BoundaryNorm([-0.5, 0.5, 1.5], cmap_cloud.N)
fig, ax = plt.subplots(figsize=(14, 6))
cloud_da.plot(ax=ax, cmap=cmap_cloud, norm=norm_cloud, add_colorbar=False)
cbar = plt.colorbar(ax.collections[0], ax=ax, ticks=[0, 1])
cbar.ax.set_yticklabels(["Clear", "Cloudy"])
ax.set_title(f"Bit 2: Cloudiness — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

pct_cloudy = cloud.sum() / cloud.size * 100
print(f"  Clear: {100 - pct_cloudy:.1f}%  Cloudy: {pct_cloudy:.1f}%")

# --- Plot 4: Bit 3 — Elevation data source ---
elev_da = xr.DataArray(elev_src, dims=et.dims, coords=et.coords)
cmap_elev = ListedColormap(["#8e44ad", "#1abc9c"])
norm_elev = BoundaryNorm([-0.5, 0.5, 1.5], cmap_elev.N)
fig, ax = plt.subplots(figsize=(14, 6))
elev_da.plot(ax=ax, cmap=cmap_elev, norm=norm_elev, add_colorbar=False)
cbar = plt.colorbar(ax.collections[0], ax=ax, ticks=[0, 1])
cbar.ax.set_yticklabels(["GMTED2010", "SRTM/other"])
ax.set_title(f"Bit 3: Elevation source — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

pct_srtm = elev_src.sum() / elev_src.size * 100
print(f"  GMTED2010: {100 - pct_srtm:.1f}%  SRTM/other: {pct_srtm:.1f}%")

# --- Plot 5: Bits 4-7 — Land cover class ---
lc_da = xr.DataArray(land_cover, dims=et.dims, coords=et.coords)
fig, ax = plt.subplots(figsize=(14, 6))
lc_da.plot(ax=ax, cmap="tab20", vmin=-0.5, vmax=15.5, add_colorbar=True)
ax.set_title(f"Bits 4-7: Land cover class — {time_label}")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

unique, counts = np.unique(land_cover, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c / land_cover.size * 100:.1f}%")


In [ ]:
# Summary stats for the selected timestep
print(f"ET_500m stats (timestep {TIMESTEP}, {time_label}):")
print(f"  min:    {float(et.min()):.1f}")
print(f"  max:    {float(et.max()):.1f}")
print(f"  mean:   {float(et.mean()):.1f}")
print(f"  NaN %:  {float(et.isnull().mean()) * 100:.1f}%")
print(f"  units:  {et.attrs.get('units', 'not set')}")
print(f"  dtype:  {et.dtype}")
print(f"  grid_mapping: {et.attrs.get('grid_mapping', 'MISSING')}")


In [ ]:
# Annual mean ET
et_annual = ds["ET_500m"].mean(dim="time")

fig, ax = plt.subplots(figsize=(12, 6))
et_annual.plot(ax=ax, cmap="YlGnBu", robust=True)
ax.set_title("Mean ET_500m — 2001 annual")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


In [ ]:
ds.close()
